CLIP embeddings for Misogyny train dataset

Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
import os
import json
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

# ===============================
# 加载 CLIP
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()
print("CLIP loaded!")

# ===============================
# 读取训练集标签
# ===============================
train_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/train"
train_df = pd.read_csv("/content/drive/MyDrive/MyThesis2026/data/cindy/data/train.csv")

print(f"训练集总数: {len(train_df)}")

# ===============================
# 生成 embeddings
# ===============================
embeddings = []
valid_filenames = []
valid_labels = []

print("生成训练集 embeddings...")
for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    filename = row["filename"]
    label = row["label"]
    img_path = os.path.join(train_image_dir, filename)

    if not os.path.exists(img_path):
        continue

    try:
        image = Image.open(img_path).convert("RGB")
        inputs = clip_processor(images=image, return_tensors="pt")
        with torch.no_grad():
            outputs = clip_model.vision_model(**inputs)
            emb = outputs.pooler_output
            emb = emb / emb.norm(dim=-1, keepdim=True)
        embeddings.append(emb.squeeze().numpy())
        valid_filenames.append(filename)
        valid_labels.append(label)
    except Exception as e:
        print(f"跳过 {filename}: {e}")

embeddings = np.array(embeddings)
print(f"完成！共 {len(embeddings)} 个 embeddings")

# 保存
np.save("/content/drive/MyDrive/misogyny_train_embeddings.npy", embeddings)
meta = {"filenames": valid_filenames, "labels": valid_labels}
with open("/content/drive/MyDrive/misogyny_train_meta.json", "w") as f:
    json.dump(meta, f)
print("Embeddings 已保存！")

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP loaded!
训练集总数: 1190
生成训练集 embeddings...


100%|██████████| 1190/1190 [13:58<00:00,  1.42it/s]

完成！共 1190 个 embeddings
Embeddings 已保存！


Install independencies

In [6]:
!pip install anthropic --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 17.7 MB/s eta 0:00:00


Test

In [7]:
import anthropic

client = userdata.get('GOOGLE_API_KEY')

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=10,
    messages=[{"role": "user", "content": "Say hello."}]
)
print(response.content[0].text)

Hello! 👋 How can I help


For one image

In [9]:
import anthropic
import base64
import re
import os
import pandas as pd

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# 读取测试集
test_df = pd.read_csv("/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv")

# 取第一张图片测试
row = test_df.iloc[0]
img_name = row["filename"]
img_path = f"/content/drive/MyDrive/MyThesis2026/data/cindy/images/test/{img_name}"

with open(img_path, "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": image_data
                    }
                },
                {
                    "type": "text",
                    "text": prompt_text
                }
            ]
        }
    ]
)

raw = response.content[0].text.strip()
print("RAW OUTPUT:", raw)

m = re.search(r"Class labels?:\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
label = m.group(1) if m else None
print("PARSED LABEL:", label)
print("TRUE LABEL:", "Misogyny" if row["label"] == 1 else "Non_Misogyny")

RAW OUTPUT: # Classification Analysis

**Step 1: Input Analysis**

Image: A cute cartoon pig character wearing a blue helmet/hat with yellow eyes
Text: "光顾着上学，忘记上吊了" (Roughly translates to: "Was so focused on studying, forgot to hang myself")

**Step 2: Misogyny Assessment**

The text contains a dark/morbid joke about suicide, but it does not:
- Target women specifically
- Contain gender-based stereotypes
- Degrade or demean women as a group
- Reference gender roles or sexist tropes

The meme appears to be a general self-deprecating joke about academic stress that could apply to any person regardless of gender.

**Step 3: Conclusion**

---

**Class labels:** Non_Misogyny

**Thought:** While the meme contains dark humor regarding suicide and mental health, it does not contain any misogynistic elements. There are no negative, insulting, stereotyping, or degrading references specifically directed at women or women as a group. The morbid joke about academic pressure is gender-neutral in na

For multi images

In [11]:
import os
import json
import base64
import re
import time
import anthropic
import pandas as pd
from tqdm import tqdm

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/ClaudeHaiku_Misogyny_ZeroShot_pred.json"

test_df = pd.read_csv(test_csv)

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode("utf-8")

def call_with_retry(image_data, media_type, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt_text
                            }
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30 * (attempt + 1)
                print(f"  限速，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

# ===============================
# 批量推理
# ===============================
for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        ext = img_name.lower().split(".")[-1]
        media_type = "image/png" if ext == "png" else "image/gif" if ext == "gif" else "image/jpeg"

        image_data = encode_image(img_path)
        raw = call_with_retry(image_data, media_type)
        label = parse_label(raw)

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 0 条，从断点继续...
剩余待处理: 340 张


推理进度:   0%|          | 0/340 [00:00<?, ?it/s]

✅ 1582.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度:   0%|          | 1/340 [00:05<28:23,  5.02s/it]

✅ 1305.jpg -> Non_Misogyny (真实: Non_Misogyny)


...

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)


推理进度: 100%|█████████▉| 339/340 [29:06<00:05,  5.48s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度: 100%|██████████| 340/340 [29:12<00:00,  5.15s/it]


完成！共 340 条结果已保存
  ERROR: 23
  Misogyny: 95
  Non_Misogyny: 222


In [12]:
import json

with open("/content/drive/MyDrive/ClaudeHaiku_Misogyny_ZeroShot_pred.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

errors = [p for p in predictions if p["predicted_label"] == "ERROR"]
print(f"ERROR 数量: {len(errors)}")
for e in errors[:3]:
    print(f"\n图片: {e['image_name']}")
    print(f"错误: {e['raw_output'][:200]}")

ERROR 数量: 23

图片: 820.jpg
错误: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.image.source.base64: The image was specified using the image/jpeg media type, but the im

图片: 149.jpg
错误: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.image.source.base64: The image was specified using the image/jpeg media type, but the im

图片: 238.jpg
错误: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.image.source.base64: The image was specified using the image/jpeg media type, but the im


Re-run error pics

In [13]:
import os
import json
import base64
import re
import time
import anthropic
import pandas as pd
from PIL import Image
from tqdm import tqdm
import io

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/ClaudeHaiku_Misogyny_ZeroShot_pred.json"

test_df = pd.read_csv(test_csv)

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def encode_image(image_path):
    # 用 PIL 检测真实格式并转换
    image = Image.open(image_path).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    return base64.standard_b64encode(buffer.read()).decode("utf-8"), "image/jpeg"

def call_with_retry(image_data, media_type, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt_text
                            }
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30 * (attempt + 1)
                print(f"  限速，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑，ERROR 的重跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        image_data, media_type = encode_image(img_path)
        raw = call_with_retry(image_data, media_type)
        label = parse_label(raw)

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

发现已有成功结果 317 条，从断点继续...
剩余待处理: 23 张


推理进度:   0%|          | 0/23 [00:00<?, ?it/s]

✅ 820.jpg -> Non_Misogyny (真实: Misogyny)


推理进度:   4%|▍         | 1/23 [00:04<01:28,  4.03s/it]

✅ 149.jpg -> Misogyny (真实: Misogyny)


...

✅ 1107.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度:  96%|█████████▌| 22/23 [01:38<00:04,  4.56s/it]

✅ 583.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度: 100%|██████████| 23/23 [01:42<00:00,  4.47s/it]


完成！共 340 条结果已保存
  Misogyny: 101
  Non_Misogyny: 239


Calculate Zero-shot result

In [14]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ===============================
# 1️⃣ 读取预测结果
# ===============================
output_json = "/content/drive/MyDrive/ClaudeHaiku_Misogyny_ZeroShot_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

# ===============================
# 2️⃣ 标签标准化
# ===============================
def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    if x == "non_misogyny":
        return "non_misogyny"
    return "non_misogyny"  # UNKNOWN 归为 non_misogyny

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

# ===============================
# 3️⃣ 计算指标
# ===============================
ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.7618
MP   : 0.7191
MR   : 0.7154
MF1  : 0.7172
WP   : 0.7599
WR   : 0.7618
WF1  : 0.7608
-------------------------------------------------

Confusion Matrix:
[[ 62  42]
 [ 39 197]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.61      0.60      0.60       104
non_misogyny       0.82      0.83      0.83       236

    accuracy                           0.76       340
   macro avg       0.72      0.72      0.72       340
weighted avg       0.76      0.76      0.76       340



Few-shot with multiple pics

In [15]:
import os
import json
import base64
import re
import time
import numpy as np
import torch
import anthropic
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import io

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
train_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/train"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/ClaudeHaiku_Misogyny_FewShot_RAG_pred.json"

test_df = pd.read_csv(test_csv)

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/misogyny_train_embeddings.npy")
with open("/content/drive/MyDrive/misogyny_train_meta.json", "r") as f:
    train_meta = json.load(f)

train_filenames = train_meta["filenames"]
train_labels = train_meta["labels"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in [0, 1]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {0: "Non_Misogyny", 1: "Misogyny"}
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Below are 2 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text, using the provided examples as reference to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

# ===============================
# 图片编码函数
# ===============================
def encode_image(image_path):
    image = Image.open(image_path).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    return base64.standard_b64encode(buffer.read()).decode("utf-8"), "image/jpeg"

# ===============================
# API 调用（带重试）
# ===============================
def call_with_retry(image_data, media_type, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=500,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image",
                                "source": {
                                    "type": "base64",
                                    "media_type": media_type,
                                    "data": image_data
                                }
                            },
                            {
                                "type": "text",
                                "text": prompt
                            }
                        ]
                    }
                ]
            )
            return response.content[0].text.strip()
        except Exception as e:
            if "429" in str(e) or "overloaded" in str(e).lower():
                wait = 30 * (attempt + 1)
                print(f"  限速，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry", "i can't help"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# ===============================
# 断点续跑
# ===============================
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

# ===============================
# 批量推理
# ===============================
for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        # RAG 检索
        test_emb = get_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        image_data, media_type = encode_image(img_path)
        raw = call_with_retry(image_data, media_type, prompt_text)
        label = parse_label(raw)

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(3)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


训练集 embeddings 加载完成，共 1190 条
没有已有结果，从头开始...
剩余待处理: 340 张


推理进度:   0%|          | 0/340 [00:00<?, ?it/s]

...

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)


推理进度: 100%|█████████▉| 339/340 [29:16<00:05,  5.06s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度: 100%|██████████| 340/340 [29:21<00:00,  5.18s/it]


完成！共 340 条结果已保存
  Misogyny: 99
  Non_Misogyny: 241


 Calculation of Few shot

In [16]:
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ===============================
# 1️⃣ 读取预测结果
# ===============================
output_json = "/content/drive/MyDrive/ClaudeHaiku_Misogyny_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

# ===============================
# 2️⃣ 标签标准化
# ===============================
def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    if x == "non_misogyny":
        return "non_misogyny"
    return "non_misogyny"

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

# ===============================
# 3️⃣ 计算指标
# ===============================
ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.7441
MP   : 0.6975
MR   : 0.6920
MF1  : 0.6945
WP   : 0.7408
WR   : 0.7441
WF1  : 0.7423
-------------------------------------------------

Confusion Matrix:
[[ 58  46]
 [ 41 195]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.59      0.56      0.57       104
non_misogyny       0.81      0.83      0.82       236

    accuracy                           0.74       340
   macro avg       0.70      0.69      0.69       340
weighted avg       0.74      0.74      0.74       340

